# DataVine Analytics — Summative Lab
### Classification, Recommendation, and Clustering with the Wine, Chickwts, and USArrests Datasets

**Group:** 8  



1. Data preparation (loading, cleaning, standardizing)
2. Dimensionality reduction (PCA)
3. Model implementation and hyperparameter tuning (k-NN, cosine-similarity recommender, K-Means & GMM)
4. Evaluation, visualization, and business interpretation



In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (classification_report, accuracy_score,
                             confusion_matrix, silhouette_score)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## Step 1: Load and Prepare Datasets
We load all three datasets.
Inspect them for missing values and standardize numerical features.


### 1.1 Wine Dataset (Classification)

In [4]:
wine_data = load_wine()
wine_df = pd.DataFrame(wine_data.data, columns=wine_data.feature_names)
wine_df['target'] = wine_data.target
wine_df['target_name'] = wine_df['target'].map(dict(enumerate(wine_data.target_names)))

print('Wine dataset shape:', wine_df.shape)
print('\nMissing values per column:')
print(wine_df.isnull().sum())
print('\nClass distribution:')
print(wine_df['target_name'].value_counts())
wine_df.describe().T

Wine dataset shape: (178, 15)

Missing values per column:
alcohol                         0
malic_acid                      0
ash                             0
alcalinity_of_ash               0
magnesium                       0
total_phenols                   0
flavanoids                      0
nonflavanoid_phenols            0
proanthocyanins                 0
color_intensity                 0
hue                             0
od280/od315_of_diluted_wines    0
proline                         0
target                          0
target_name                     0
dtype: int64

Class distribution:
target_name
class_1    71
class_0    59
class_2    48
Name: count, dtype: int64


,count,mean,std,min,25%,50%,75%,max
alcohol,178.0,13.000618,0.811827,11.03,12.3625,13.050,13.6775,14.83
malic_acid,178.0,2.336348,1.117146,0.74,1.6025,1.865,3.0825,5.80
ash,178.0,2.366517,0.274344,1.36,2.2100,2.360,2.5575,3.23
alcalinity_of_ash,178.0,19.494944,3.339564,10.60,17.2000,19.500,21.5000,30.00
magnesium,178.0,99.741573,14.282484,70.00,88.0000,98.000,107.0000,162.00
total_phenols,178.0,2.295112,0.625851,0.98,1.7425,2.355,2.8000,3.88
flavanoids,178.0,2.029270,0.998859,0.34,1.2050,2.135,2.8750,5.08
nonflavanoid_phenols,178.0,0.361854,0.124453,0.13,0.2700,0.340,0.4375,0.66
proanthocyanins,178.0,1.590899,0.572359,0.41,1.2500,1.555,1.9500,3.58
color_intensity,178.0,5.058090,2.318286,1.28,3.2200,4.690,6.2000,13.00


### 1.2 Chickwts Dataset (Recommendation System)

In [5]:
chickwts_url = 'https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/datasets/chickwts.csv'
chickwts_df = pd.read_csv(chickwts_url, index_col=0)

print('Chickwts dataset shape:', chickwts_df.shape)
print('\nMissing values per column:')
print(chickwts_df.isnull().sum())
print('\nFeed types:', chickwts_df['feed'].unique())
chickwts_df.head()

Chickwts dataset shape: (71, 2)

Missing values per column:
weight    0
feed      0
dtype: int64

Feed types: <StringArray>
['horsebean', 'linseed', 'soybean', 'sunflower', 'meatmeal', 'casein']
Length: 6, dtype: str


,weight,feed
rownames,,
1,179,horsebean
2,160,horsebean
3,136,horsebean
4,227,horsebean
5,217,horsebean


### 1.3 USArrests Dataset (Clustering)

In [6]:
usarrests_url = 'https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/datasets/USArrests.csv'
usarrests_df = pd.read_csv(usarrests_url, index_col=0)
usarrests_df.index.name = 'State'

print('USArrests dataset shape:', usarrests_df.shape)
print('\nMissing values per column:')
print(usarrests_df.isnull().sum())
usarrests_df.describe().T

USArrests dataset shape: (50, 4)

Missing values per column:
Murder      0
Assault     0
UrbanPop    0
Rape        0
dtype: int64


,count,mean,std,min,25%,50%,75%,max
Murder,50.0,7.788,4.355510,0.8,4.075,7.25,11.250,17.4
Assault,50.0,170.760,83.337661,45.0,109.000,159.00,249.000,337.0
UrbanPop,50.0,65.540,14.474763,32.0,54.500,66.00,77.750,91.0
Rape,50.0,21.232,9.366385,7.3,15.075,20.10,26.175,46.0
